# Ortholog Lookup - Mouse-anchored STAR, NO correlation threshold (Python)

Variant of `ortholog_lookup_star_prateek.ipynb`. Everything is identical **except the 0.6
correlation *value* threshold is removed**: for a multi-ortholog group, instead of requiring the
best candidate to clear a 0.6 Pearson threshold, we simply **keep the member of the group with the
highest correlation to mouse**, whatever that correlation value is. No gene is dropped for being
"sub-threshold".

The **per-cell detection floor is retained** at the same level as `ortholog_lookup_star_prateek.ipynb`
(`DETECT_FLOOR = 0.0001`, i.e. 0.01%): a gene detected in fewer than 0.01% of a species' cells is
still not correlation-eligible. So only the 0.6 *value* gate is dropped, not the detection gate.

As before, every comparison is **mouse vs species X only** (both name and correlation), and a
gene is kept iff it is strict 1:1 to the mouse gene in all five MM-X pairs.

Priority inside `find_homolog('MM', X)` is unchanged:
1. exact gene symbol,
2. a *unique* ohnolog-suffixed name (Bach2 -> bach2a),
3. **highest correlation** among the group's candidates that pass the detection floor -- now with
   **no 0.6 value threshold**.

> Expected effect vs the thresholded star: strictly **more** genes captured, because the only
> reason a correlation-resolved pair was previously dropped (best in-floor candidate < 0.6) no
> longer applies. Genes still drop only when (a) a species has no ortholog in the group, (b) the
> best-correlated candidate collides with another mouse gene in the strict 1:1 step (e.g. a
> paralog like Nfix -> nfia), or (c) no candidate has computable, in-floor expression to correlate.

In [5]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad

ORTHO_PATH = '../../Vert_emapper_allorgs_07032026.tsv'
HUB = 'MM'                                          # mouse is the anchor
organisms = ['MM', 'MO', 'CJ', 'AC', 'XT', 'DR']   # MM (mouse) first
OTHERS = [o for o in organisms if o != HUB]
DETECT_FLOOR = 0.0001                               # detection floor restored (matches ortholog_lookup_star_prateek.ipynb):
                                                    # a gene detected in <0.01% of a species' cells is not correlation-eligible
# NOTE: no CORR_THRESHOLD -- the 0.6 correlation gate is still removed; ONLY the detection floor is restored.

H5ADS = {
    'MM': '../../Active_SAM_joined/SAM_Allen_Insitute_NN_subclass_250.h5ad',
    'MO': '../../Active_SAM_joined/SAM_MO_soupx_plus5_cleaned_03122025.h5ad',
    'CJ': '../../Active_SAM_joined/SAM_CJ_joined_v2_cleaned_03122025.h5ad',
    'AC': '../../Active_SAM_joined/SAM_AC_ncbi_soupx_cleaned_03122025.h5ad',
    'XT': '../../Active_SAM_joined/SAM_XT_joined_Slc17a6_cleaned_03122205.h5ad',
    'DR': '../../Active_SAM_joined/SAM_DR_ncbi_joined_cleaned_07172026.h5ad',
}
CELLTYPE_COL = {
    'MM': 'subclass_id_label',
    'MO': 'ss_subclass_nounlabeled_03102026',
    'CJ': 'ss_subclass_nounlabeled_nmm_v4_nn',
    'AC': 'ss_subclass_nounlabeled_nmm_v4_nn',
    'XT': 'ss_subclass_nounlabeled_nmm_v4_nn',
    'DR': 'ss_subclass_nounlabeled_nmm_v4_nn',
}

## 1. Load the orthogroup table

In [6]:
# index = eggNOG OG id; cells: '' = no gene, 'g1,g2' = multiple orthologs
ortho = pd.read_csv(ORTHO_PATH, sep='\t', index_col=0, dtype=str)
ortho = ortho[organisms].fillna('').apply(lambda s: s.str.strip())
print('orthogroups:', ortho.shape)
for o in organisms:
    n_any = (ortho[o] != '').sum()
    n_multi = ortho[o].str.contains(',').sum()
    print(f'  {o}: {n_any:6d} rows with a gene, {n_multi:5d} multi-ortholog')

orthogroups: (22547, 6)
  MM:  16847 rows with a gene,  1683 multi-ortholog
  MO:  16105 rows with a gene,  1971 multi-ortholog
  CJ:  13780 rows with a gene,  1300 multi-ortholog
  AC:  15464 rows with a gene,  2187 multi-ortholog
  XT:  14510 rows with a gene,  1830 multi-ortholog
  DR:  16847 rows with a gene,  7163 multi-ortholog


## 2. Per-subclass mean expression per species

Aggregates `adata.X` by the species' cell-type column into a genes x subclass matrix.
The star only ever correlates **MM vs X**, so the diagnostic reports shared subclass labels
between mouse and each other species.

> The SAM `.X` is already normalized, so per-subclass mean expression is taken as-is.

In [7]:
def subclass_mean(adata, col):
    """Return a genes x subclass DataFrame of mean expression (data is already normalized)."""
    labels = adata.obs[col].astype(str).values
    keep = ~pd.isna(adata.obs[col].values) & ~np.isin(labels, ['nan', '', 'unlabeled', 'NA'])
    labels = labels[keep]
    X = adata.X[keep]
    cats, codes = np.unique(labels, return_inverse=True)
    ind = sp.csr_matrix((np.ones(len(codes)), (np.arange(len(codes)), codes)),
                        shape=(len(codes), len(cats)))
    counts = np.asarray(ind.sum(axis=0)).ravel()
    sums = (X.T @ ind)
    sums = np.asarray(sums.todense()) if sp.issparse(sums) else np.asarray(sums)
    means = sums / counts[None, :]
    df = pd.DataFrame(means, index=np.asarray(adata.var_names, dtype=str), columns=cats)
    df = df[~df.index.duplicated(keep='first')]  # correlation lookup needs unique gene index
    return df

means, detected = {}, {}
for o in organisms:
    a = ad.read_h5ad(H5ADS[o])
    col = CELLTYPE_COL[o]
    means[o] = subclass_mean(a, col)
    # per-cell detection fraction on the same labeled cells used for the means (restored floor)
    labels = a.obs[col].astype(str).values
    keep = ~pd.isna(a.obs[col].values) & ~np.isin(labels, ['nan', '', 'unlabeled', 'NA'])
    X = a.X[keep]
    frac = np.asarray((X > 0).sum(axis=0)).ravel() / X.shape[0]
    d = pd.Series(frac, index=np.asarray(a.var_names, dtype=str))
    detected[o] = d[~d.index.duplicated(keep='first')]  # match the unique gene index of means
    print(f'{o}: {means[o].shape[0]} genes x {means[o].shape[1]} subclasses; '
          f'{int((detected[o] >= DETECT_FLOOR).sum())} genes pass {DETECT_FLOOR:.2%} detection floor')
    del a

MM: 32285 genes x 334 subclasses; 25302 genes pass 0.01% detection floor
MO: 19307 genes x 146 subclasses; 17975 genes pass 0.01% detection floor
CJ: 20032 genes x 136 subclasses; 18258 genes pass 0.01% detection floor
AC: 23671 genes x 127 subclasses; 21852 genes pass 0.01% detection floor
XT: 19621 genes x 120 subclasses; 18325 genes pass 0.01% detection floor
DR: 25503 genes x 61 subclasses; 21557 genes pass 0.01% detection floor


In [8]:
print('shared subclass labels (MM vs each species):')
for x in OTHERS:
    shared = means[HUB].columns.intersection(means[x].columns)
    print(f'  MM-{x}: {len(shared)}')

shared subclass labels (MM vs each species):
  MM-MO: 115
  MM-CJ: 98
  MM-AC: 93
  MM-XT: 79
  MM-DR: 31


## 3. FindHomolog: resolve each MM-X pair to strict 1:1 (best correlation, NO threshold)

In [9]:
# Teleost ohnolog suffixes (Bach2 -> bach2a/bach2b). Used only to extend the NAME match.
_OHNOLOG_SUFFIXES = ('a', 'b')

def _ohnolog_match(g1, g2):
    """True if g2 == g1 or is g1 + an ohnolog suffix (case-insensitive)."""
    a, b = str(g1).lower(), str(g2).lower()
    if a == b:
        return True
    if len(a) < 3:
        return False
    return b.startswith(a) and b[len(a):] in _OHNOLOG_SUFFIXES

def _corr(g1, g2, m1, m2, shared, d1=None, d2=None, floor=0.0):
    """Pearson correlation of per-subclass mean expression, with the per-cell DETECTION GATE
    restored (matches ortholog_lookup_star_prateek.ipynb): a gene detected in fewer than
    `floor` of its species' cells is not correlation-eligible (returns nan). The 0.6 value
    threshold is still NOT applied here -- only the detection floor is."""
    if d1 is not None and float(d1.get(g1, 0.0)) < floor:
        return np.nan
    if d2 is not None and float(d2.get(g2, 0.0)) < floor:
        return np.nan
    if g1 not in m1.index or g2 not in m2.index:
        return np.nan
    v1 = m1.loc[g1, shared].to_numpy(dtype=float)
    v2 = m2.loc[g2, shared].to_numpy(dtype=float)
    if v1.std() == 0 or v2.std() == 0:
        return np.nan
    return np.corrcoef(v1, v2)[0, 1]

def _resolve_one_to_one(oto, o1, o2, m1, m2, shared, d1=None, d2=None):
    """Greedy strict 1:1. Priority: exact symbol > ohnolog name > correlation. The correlation
    score inherits the detection gate via _corr."""
    oto = oto.drop_duplicates([o1, o2]).copy()
    def score(g1, g2):
        if str(g1).lower() == str(g2).lower():
            return 3.0                       # exact symbol wins
        if _ohnolog_match(g1, g2):
            return 2.0                       # ohnolog name match
        if m1 is not None:
            c = _corr(g1, g2, m1, m2, shared, d1, d2, DETECT_FLOOR)
            return c if c == c else -1.0     # correlation (<=1) or nan/undetected -> deprioritize
        return 0.0
    oto['_s'] = [score(a, b) for a, b in zip(oto[o1], oto[o2])]
    oto = oto.sort_values('_s', ascending=False, kind='mergesort')
    oto = oto.drop_duplicates(subset=[o1], keep='first')
    oto = oto.drop_duplicates(subset=[o2], keep='first')
    return oto.drop(columns='_s').reset_index(drop=True)

def find_homolog(o1, o2, ortho, m1=None, m2=None, d1=None, d2=None, verbose=True):
    """Resolve o1-o2 to strict 1:1. In the star, o1 is always the hub (MM). NO 0.6 correlation
    threshold: the best-correlated candidate is always taken (when a correlation is computable),
    but the per-cell detection floor is applied via _corr (d1/d2 are the detection fractions)."""
    shared = m1.columns.intersection(m2.columns) if (m1 is not None and m2 is not None) else None
    df = ortho.loc[(ortho[o1] != '') & (ortho[o2] != ''), [o1, o2]].copy()
    df[o1] = df[o1].str.split(',')
    df = df.explode(o1)
    df[o1] = df[o1].str.strip()
    single2 = ~df[o2].str.contains(',')
    oto = df.loc[single2, [o1, o2]].copy()          # org2 already single
    totest = df.loc[~single2, [o1, o2]].copy()      # org2 is a candidate list (within this OG)
    # 2. name match (exact symbol, or a UNIQUE ohnolog-suffixed candidate)
    name_rows, still = [], []
    for g1, cand in zip(totest[o1], totest[o2]):
        cands = [c.strip() for c in cand.split(',')]
        exact = [c for c in cands if c.lower() == g1.lower()]
        if exact:
            name_rows.append((g1, exact[0]))
            continue
        ohno = [c for c in cands if _ohnolog_match(g1, c)]
        if len(ohno) == 1:
            name_rows.append((g1, ohno[0]))
        else:
            still.append((g1, cand))
    oto = pd.concat([pd.DataFrame(name_rows, columns=[o1, o2]), oto], ignore_index=True)
    totest = pd.DataFrame(still, columns=[o1, o2])
    # 3. correlation -- NO VALUE THRESHOLD, but detection-floor gated: take the highest-correlated
    #    candidate whatever its value, among candidates that pass the detection floor (the gate
    #    lives in _corr). A pair is skipped only if NO candidate has a computable, in-floor corr.
    if m1 is not None and len(totest):
        corr_rows = []
        for g1, cand in zip(totest[o1], totest[o2]):
            cands = [c.strip() for c in cand.split(',')]
            scored = [(_corr(g1, c, m1, m2, shared, d1, d2, DETECT_FLOOR), c) for c in cands]
            scored = [(v, c) for v, c in scored if v == v]
            if scored:
                best_v, best_c = max(scored)
                corr_rows.append((g1, best_c))       # <-- 0.6 gate removed; detection floor kept
        oto = pd.concat([oto, pd.DataFrame(corr_rows, columns=[o1, o2])], ignore_index=True)
    # 4. enforce strict 1:1
    oto = _resolve_one_to_one(oto, o1, o2, m1, m2, shared, d1, d2)
    if verbose:
        print(f'{o1}-{o2}: {len(oto):6d} one-to-one '
              f'(dup {o1}={oto[o1].duplicated().sum()}, dup {o2}={oto[o2].duplicated().sum()})')
    return oto

In [10]:
# Only the five MM-X pairwise comparisons are needed for the star.
pairwise_star = {}
for x in OTHERS:
    pairwise_star[x] = find_homolog(HUB, x, ortho, means[HUB], means[x], detected[HUB], detected[x])

MM-MO:  16648 one-to-one (dup MM=0, dup MO=0)
MM-CJ:  13133 one-to-one (dup MM=0, dup CJ=0)
MM-AC:  14433 one-to-one (dup MM=0, dup AC=0)
MM-XT:  13624 one-to-one (dup MM=0, dup XT=0)
MM-DR:  13082 one-to-one (dup MM=0, dup DR=0)


## 4. Star set: join the five MM-X tables on the mouse gene

In [11]:
def star_set(pairwise_star, organisms, hub='MM'):
    """Mouse-anchored star: 1:1 to the hub for every species. Inner-join on the hub gene."""
    others = [o for o in organisms if o != hub]
    merged = pairwise_star[others[0]].copy()
    for o in others[1:]:
        merged = merged.merge(pairwise_star[o], on=hub, how='inner')
    return merged[organisms]

star = star_set(pairwise_star, organisms, HUB)
print(f'star, NO threshold (1:1 to mouse in all {len(OTHERS)} pairs): {len(star)} genes')
print('Tcf4 in star set?', 'Tcf4' in set(star[HUB]))

star, NO threshold (1:1 to mouse in all 5 pairs): 10182 genes
Tcf4 in star set? True


In [12]:
# Save the no-threshold star set
star.to_csv('../../OTO_star_nothreshold_6species_ohnlogs_expresssionthresh_07262026.tsv', sep='\t', index=False)
print('wrote OTO_star_nothreshold_6species_ohnlogs_expresssionthresh_07262026.tsv')

wrote OTO_star_nothreshold_6species_ohnlogs_expresssionthresh_07262026.tsv


## 5. Relaxed no-threshold star set: 1:1 to mouse in >=4 of the 5 non-mouse species

In [13]:
def star_missing_one(pairwise_star, organisms, hub='MM'):
    others = [o for o in organisms if o != hub]
    merged = pairwise_star[others[0]].copy()
    for o in others[1:]:
        merged = merged.merge(pairwise_star[o], on=hub, how='outer')   # outer -> keep partials
    merged = merged[organisms]
    n_missing = merged[others].isna().sum(axis=1)
    out = merged[n_missing <= 1].copy()
    out['n_species'] = len(organisms) - out[others].isna().sum(axis=1)  # 6 (complete) or 5
    return out.reset_index(drop=True)

star_relaxed = star_missing_one(pairwise_star, organisms, HUB)
complete = int((star_relaxed['n_species'] == 6).sum())
miss1    = int((star_relaxed['n_species'] == 5).sum())
print(f'star relaxed (no threshold): complete(all 6)={complete}  missing-exactly-one={miss1}  total(>=5)={len(star_relaxed)}')
m = star_relaxed[star_relaxed['n_species'] == 5]
by = m[organisms].isna().idxmax(axis=1).value_counts()
print('   missing-one by species: ' + ', '.join(f'{s}={int(by[s])}' for s in by.index))

star_relaxed.to_csv('../../OTO_star_nothreshold_missing_le1_expressionthresh_07262026.tsv', sep='\t', index=False)
print('wrote OTO_star_nothreshold_missing_le1_expressionthresh_07262026.tsv')

star relaxed (no threshold): complete(all 6)=10182  missing-exactly-one=3073  total(>=5)=13255
   missing-one by species: CJ=951, DR=928, XT=671, AC=303, MO=220
wrote OTO_star_nothreshold_missing_le1_expressionthresh_07262026.tsv


In [14]:
# Genes missing in at most 2 organisms (mouse is always the anchor, so the <=2 missing are
# non-mouse species -> present in >=3 of the 5 non-mouse species, plus mouse).
# n_species = 6 (complete), 5 (one missing), or 4 (two missing).
def star_missing_two(pairwise_star, organisms, hub='MM'):
    others = [o for o in organisms if o != hub]
    merged = pairwise_star[others[0]].copy()
    for o in others[1:]:
        merged = merged.merge(pairwise_star[o], on=hub, how='outer')   # outer -> keep partials
    merged = merged[organisms]
    n_missing = merged[others].isna().sum(axis=1)
    out = merged[n_missing <= 2].copy()
    out['n_species'] = len(organisms) - out[others].isna().sum(axis=1)  # 6, 5, or 4
    return out.reset_index(drop=True)

star_missing2 = star_missing_two(pairwise_star, organisms, HUB)
for k in (6, 5, 4):
    print(f'   n_species={k}: {int((star_missing2["n_species"] == k).sum())}')
print(f'star nothreshold missing<=2 (>=4 species incl. mouse): {len(star_missing2)} genes total')
# total absences per non-mouse species (a row missing two species is counted in both)
by = star_missing2[OTHERS].isna().sum(axis=0)
print('   absences by species: ' + ', '.join(f'{s}={int(by[s])}' for s in OTHERS))

star_missing2.to_csv('../../OTO_star_nothreshold_missing_le2_expressionthresh_07262026.tsv', sep='\t', index=False)
print('wrote OTO_star_nothreshold_missing_le2_expressionthresh_07262026.tsv')

   n_species=6: 10182
   n_species=5: 3073
   n_species=4: 1230
star nothreshold missing<=2 (>=4 species incl. mouse): 14485 genes total
   absences by species: MO=368, CJ=1591, AC=633, XT=1265, DR=1676
wrote OTO_star_nothreshold_missing_le2_expressionthresh_07262026.tsv
